# Pandas

A sales table needs to become a report that someone can trust. Begin by identifying what one row represents, then inspect missing values, derive quantities, combine reference data and reconcile the final totals. Each transformation should have an expected effect on row counts and on the meaning of a sale.

Suppose your teammate sends you a CSV full of rows and columns. It is not clean yet. Some values are missing. Some rows belong to the same customer. Some dates are strings, not dates. You need to inspect the data, filter it, group it, and reshape it before you can answer a business question.

That is exactly where Pandas becomes useful.

Pandas is the library for tabular data in Python. It gives you a way to think in rows, columns, groups, and time. If you have a spreadsheet, a CSV, a table from a database, or a dataset from a project, Pandas is usually the first tool you reach for.

The main idea is simple: treat data as a table, and then ask questions of that table.


## 1. Start with the sales decision and the meaning of a row

Before calculating anything, state whether a row is an order, a line item or a summary in this example. Decide which columns would be needed to distinguish “more sales” from “more revenue.” Keep that distinction as new columns are introduced.

Imagine we are looking at a small sales record. Each row is a transaction. Each column is a feature of that transaction.

| customer | city | amount | items |
| --- | --- | ---: | ---: |
| Alice | Delhi | 120 | 2 |
| Bob | Mumbai | 85 | 1 |
| Cia | Delhi | 140 | 3 |
| Dani | Bengaluru | 95 | 2 |

A natural question is: "Which city has higher sales?" or "Which customer ordered the most items?" To answer these, we need a structure that stores rows and columns in a predictable way.

That structure is a DataFrame.

<div style="background-color:#eaf4fb; border-left:5px solid #2878a5; padding:10px 14px; margin:12px 0;">
<strong>Start with the shape of the information.</strong><br>
A row keeps facts about one transaction together. A column lets us compare the same kind of fact across transactions. Pandas gives names to both directions, so we can ask for exactly the records or measurements we need.
</div>

| Pandas object | Shape | Example in this table | When it helps |
| --- | --- | --- | --- |
| `DataFrame` | Two-dimensional | The complete sales table | Compare or transform several columns and rows together |
| `Series` | One-dimensional | `sales['amount']` | Work with one labeled column of values |
| Index | Labels for rows | `0`, `1`, `2`, `3` by default | Keep track of which record each value belongs to |

A Series is one labeled sequence of values. Selecting one DataFrame column, such as `sales['amount']`, gives us a Series. This distinction matters because some Pandas methods work on a single Series while others combine multiple columns in a DataFrame.

In [ ]:
import pandas as pd
import numpy as np

sales = pd.DataFrame({
    'customer': ['Alice', 'Bob', 'Cia', 'Dani'],
    'city': ['Delhi', 'Mumbai', 'Delhi', 'Bengaluru'],
    'amount': [120, 85, 140, 95],
    'items': [2, 1, 3, 2],
})

sales


## 2. Look at the table before doing anything fancy

When you first get data, the first job is not modeling. It is just understanding the shape and the values.

This is where `.head()`, `.info()`, and `.describe()` help.

- `.head()` shows the top rows
- `.info()` shows column types and missing values
- `.describe()` gives a quick summary of numeric columns

The main idea is to inspect the data before making assumptions.


In [ ]:
print(sales.head())
print()
print(sales.info())
print()
print(sales.describe())


## 3. Select rows and columns carefully

Now we move from looking to asking: "What is the amount for Alice?" or "Show me the cities and amounts only?" Selection is how we point Pandas at the part of the table that contains the answer.

There are two common ways to select data:

| Selector | What the number or name means | Example | Result |
| --- | --- | --- | --- |
| `.loc` | A row or column label | `sales.loc[0, 'amount']` | Amount in the row labeled `0` |
| `.iloc` | An integer position, starting at zero | `sales.iloc[0, 2]` | Value in the first row and third column |

In this example the default row labels happen to be `0`, `1`, `2`, and `3`, so `.loc[0]` and `.iloc[0]` both select the first row. After we change the index, the labels and positions can differ. That is why `.loc` means label and `.iloc` means integer location.

<div style="background-color:#fff5df; border-left:5px solid #d28a16; padding:10px 14px; margin:12px 0;">
<strong>Watch the slice boundary.</strong><br>
With `.iloc`, the stop position is excluded: `sales.iloc[:2]` selects positions 0 and 1. With label slices in `.loc`, both endpoint labels are included when they exist. This small difference can change which records enter a calculation.
</div>

The colon means "take all values in this direction." So `sales.loc[:, ['customer', 'amount']]` keeps every row and selects just two named columns. In `sales.iloc[:2, 1:3]`, `:2` selects the first two rows and `1:3` selects columns at positions 1 and 2.

In [ ]:
sales.loc[0]


In [ ]:
sales.loc[:, ['customer', 'amount']]


In [ ]:
sales.iloc[:2, 1:3]


## 4. Filtering is just asking a true-or-false question

A filter in Pandas is usually a boolean mask: one True or False value for each row. Suppose we ask, "Which sales are above 100?" Pandas compares every amount with 100:

| customer | amount | Check: amount > 100 | Keep this row? |
| --- | ---: | --- | --- |
| Alice | 120 | `120 > 100` is True | Yes |
| Bob | 85 | `85 > 100` is False | No |
| Cia | 140 | `140 > 100` is True | Yes |
| Dani | 95 | `95 > 100` is False | No |

The expression `sales['amount'] > 100` produces the True/False Series shown in the third column. When we write `sales[mask]`, Pandas keeps the rows aligned with True values.

<div style="background-color:#eaf4fb; border-left:5px solid #2878a5; padding:10px 14px; margin:12px 0;">
<strong>Read the filter in two steps.</strong><br>
First, make a condition that answers one yes/no question per row. Then use that condition to select rows. This makes it easier to check whether a complicated filter is doing what we intended.
</div>

To combine conditions, use `&` for AND and `|` for OR, and put parentheses around each comparison, for example `(sales['amount'] > 100) & (sales['city'] == 'Delhi')`. `sort_values` changes row order; it does not filter rows.

In [ ]:
mask = sales['amount'] > 100
mask


In [ ]:
sales[mask]


In [ ]:
sales.sort_values('amount', ascending=False)


## 5. Add new information to the table

Once the data is in a DataFrame, we often create new derived columns. This is how we turn raw values into useful signals.

For example, we can compute a new column called `amount_per_item` by dividing `amount` by `items`.

This step is important because the original table may not answer the question directly, but a derived column can.


In [ ]:
sales['amount_per_item'] = sales['amount'] / sales['items']
sales


In [ ]:
sales = sales.assign(
    high_value=sales['amount'] > 100,
    city_group=sales['city'].str.upper(),
)
sales[['customer', 'amount', 'high_value', 'city_group']]


## 6. Real data is imperfect: missing values

A dataset is rarely perfect. Some rows may have missing values, and we need to decide what to do with them.

Pandas gives us tools to detect and handle missing data:

- `.isna()` : mark missing positions
- `.fillna(...)` : replace missing values
- `.dropna()` : remove rows with missing values

The decision depends on the problem, but the important part is to be explicit.


In [ ]:
missing = pd.DataFrame({
    'name': ['Asha', 'Ravi', 'Mina', 'Leo'],
    'score': [90, np.nan, 85, 72],
    'team': ['A', 'B', None, 'B'],
})

print(missing.isna())
print()
print(missing.fillna({'score': missing['score'].mean(), 'team': 'unknown'}))
print()
print(missing.dropna())


## 7. Make the grouping match the comparison

Predict the number and identity of groups before aggregation. Check one group manually and compare the sum of group totals with the overall total. A mean of group means does not necessarily equal the mean across all rows.

The next common task is grouping. Instead of looking at each row individually, we ask: "What is the pattern by category?" For the student table, we might compare average scores by team.

`groupby('team')` temporarily separates the rows by team. `.agg(...)` calculates requested statistics for each group, and `.reset_index()` turns the group labels back into a regular column.

| Team A scores | Team B scores | Summary question | Calculation |
| --- | --- | --- | --- |
| 88, 91, 95 | 76, 68 | Average score for each team? | Add a team's scores and divide by its number of students |
| 88, 91, 95 | 76, 68 | Highest score for each team? | Select the largest score in each group |
| 3 students | 2 students | How many rows per team? | Count names with `size` |

<div style="background-color:#eaf4fb; border-left:5px solid #2878a5; padding:10px 14px; margin:12px 0;">
<strong>Grouping changes the unit of the answer.</strong><br>
Before grouping, one row represents one student. After grouping by team, one output row represents one team. That is why the resulting summary is shorter than the original table.
</div>

The code below asks for three separate summaries: row count, average score, and maximum score. Named aggregation makes both the input column and the output column name explicit.

In [ ]:
students = pd.DataFrame({
    'name': ['Asha', 'Ravi', 'Mina', 'Leo', 'Sara'],
    'team': ['A', 'B', 'A', 'B', 'A'],
    'score': [88, 76, 91, 68, 95],
    'attendance': [96, 89, 93, 74, 98],
})

students.groupby('team').agg(
    rows=('name', 'size'),
    avg_score=('score', 'mean'),
    max_score=('score', 'max'),
).reset_index()


## 8. Merge tables when the information is split across two tables

Sometimes information is spread across multiple tables. One may contain student marks, while another contains attendance. A merge matches rows using a shared key, here `name`, so the information for Asha stays together rather than being combined with another student's row.

| `how` choice | Rows retained | Use it when |
| --- | --- | --- |
| `left` | Every row from the left table, with matching right-side values | The left table defines the records we must keep |
| `inner` | Only keys found in both tables | We only want records with a match on both sides |
| `right` | Every row from the right table | The right table defines the records we must keep |
| `outer` | Keys from either table | We need to inspect all records, including unmatched ones |

The example uses `how='left'`: marks is the left table, so all students with marks stay in the output. The `on='name'` argument tells Pandas which column is the matching key.

<div style="background-color:#fff5df; border-left:5px solid #d28a16; padding:10px 14px; margin:12px 0;">
<strong>Repeated keys can multiply rows.</strong><br>
If a name appears twice in each input, matching every occurrence can produce four output rows for that name. Real datasets often use unique IDs instead of names; check key uniqueness when row counts matter.
</div>

In [ ]:
marks = pd.DataFrame({
    'name': ['Asha', 'Ravi', 'Mina', 'Leo'],
    'math': [92, 78, 88, 70],
    'science': [90, 80, 94, 72],
})

attendance = pd.DataFrame({
    'name': ['Asha', 'Ravi', 'Mina', 'Leo'],
    'attendance': [96, 89, 93, 74],
})

marks.merge(attendance, on='name', how='left')


## 9. Reshape the table when the question changes

Sometimes the table is not in the shape you need. One shape is good for entering data, but another is better for analysis.

Two important patterns are:

- `pivot_table()`: turn rows into columns
- `melt()`: turn columns into rows

This is not random. It is about making the table match the question.


In [ ]:
sales_long = pd.DataFrame({
    'month': ['Jan', 'Jan', 'Feb', 'Feb'],
    'product': ['A', 'B', 'A', 'B'],
    'sales': [120, 90, 140, 110],
})

sales_long.pivot_table(index='month', columns='product', values='sales', aggfunc='sum')


In [ ]:
df_melt = pd.DataFrame({
    'month': ['Jan', 'Feb'],
    'A': [120, 140],
    'B': [90, 110],
})

df_melt.melt(id_vars='month', value_vars=['A', 'B'], var_name='product', value_name='sales')


## 10. Time is a special kind of data

Dates stored as strings look readable, but text does not know that one date comes before another. Convert the column to a datetime type before doing calendar operations.

| Step | Pandas operation | Why it matters |
| --- | --- | --- |
| Parse text as timestamps | `pd.to_datetime(daily['timestamp'])` | Enables date-aware sorting and comparisons |
| Set a time index | `.set_index('timestamp')` | Gives resampling a time axis to use |
| Create daily bins | `.resample('D')` | Groups observations into one-day intervals |
| Aggregate each bin | `.sum()` | Adds the values that fall within each interval |

The sample has no measurement on January 4. Resampling to daily frequency creates that date in the output; its sum is zero because the default sum of an empty bin is zero. That zero means "no values were summed in this bin," not "a sensor measured zero."

<div style="background-color:#fff5df; border-left:5px solid #d28a16; padding:10px 14px; margin:12px 0;">
<strong>Choose the resampling rule and aggregation together.</strong><br>
`'D'` means daily bins, but `.sum()` is only right when adding measurements makes sense. For rates or temperatures, a mean or another statistic may be more meaningful; missing bins may need to remain missing instead of becoming zero.
</div>

In [ ]:
daily = pd.DataFrame({
    'timestamp': ['2024-01-01', '2024-01-02', '2024-01-03', '2024-01-05', '2024-01-10'],
    'value': [10, 12, 15, 13, 18],
})

daily['timestamp'] = pd.to_datetime(daily['timestamp'])
daily.set_index('timestamp').resample('D').sum()


## 11. `map`, `apply`, and string methods

Sometimes vectorized operations are elegant, but not every task can be written in one clean formula. In those cases, we use Python functions on values.

- `map()` : transform each value in a Series
- `apply()` : apply a function to each row or column
- `.str` methods : operate on string columns

These are especially useful when the logic is more expressive as Python code than as a formula.


In [ ]:
students['score_label'] = students['score'].map(lambda x: 'excellent' if x >= 90 else 'good')
students['attendance_flag'] = students['attendance'].apply(lambda x: x >= 90)
students[['name', 'score', 'score_label', 'attendance_flag']]


In [ ]:
names = pd.Series(['asha', 'ravi', 'mina', 'leo'])
names.str.title()


## 12. Read data from files

Most real work begins with a file. Pandas can read CSVs, Excel files, JSON, and many other formats directly.

The key idea is that the file is not the end of the story. It is the start of the workflow: read → inspect → clean → summarize → model.

A simple example is reading a tab-separated file.


In [ ]:
from pathlib import Path

csv_df = pd.read_csv(Path('data') / 'sales_sample.csv')
csv_df.head()


This CSV is a small synthetic sales dataset included in `data/sales_sample.csv`. It works offline. For your own data, replace the path and specify options such as `sep`, `dtype`, and `parse_dates` to match the file.


## 13. Detect and remove duplicate records

Duplicate rows can inflate totals. Choose the columns that define one business record before dropping duplicates; matching customer names alone do not mean two orders are the same.

In [ ]:
orders = pd.DataFrame({
    'order_id': [101, 102, 102, 103, 104],
    'date': ['2024-01-01', '2024-01-02', '2024-01-02', '2024-01-03', '2024-01-04'],
    'customer_id': [1, 2, 2, 1, 3],
    'amount': [120, 80, 80, 160, 40],
})
print(orders.loc[orders.duplicated('order_id', keep=False)])
clean_orders = orders.drop_duplicates('order_id').copy()
assert clean_orders['order_id'].is_unique
clean_orders


## 14. Rolling windows and group transformations

A rolling window summarizes neighboring observations. Sort by time first. Here each day has one order, so a three-row window is also three days; for irregular dates, use a time-based window such as `rolling('3D')`. `transform` returns one value per original row, which makes group comparisons easy.

In [ ]:
clean_orders['date'] = pd.to_datetime(clean_orders['date'])
daily_revenue = clean_orders.set_index('date')['amount'].sort_index()
print(pd.DataFrame({'revenue': daily_revenue,
                    'three_day_mean': daily_revenue.rolling('3D', min_periods=1).mean()}))
clean_orders['customer_total'] = clean_orders.groupby('customer_id')['amount'].transform('sum')
clean_orders


## 15. Use categorical and nullable types

Categorical columns represent a known set of labels and can define a meaningful order. Nullable integer types preserve missing values without converting integers to floats. For large repeated text columns, categorical storage can reduce memory; measure it on your actual data.

In [ ]:
tiers = pd.CategoricalDtype(categories=['bronze', 'silver', 'gold'], ordered=True)
customers = pd.DataFrame({
    'customer_id': [1, 2, 3],
    'tier': pd.Series(['gold', 'silver', 'bronze'], dtype=tiers),
    'visits': pd.Series([5, None, 1], dtype='Int64'),
})
print(customers.dtypes)
customers.sort_values('tier')


## 16. Find unmatched keys before accepting a merged report

Predict whether this join should preserve row count. Inspect unmatched keys and the validation rule, then explain how a duplicate reference key could inflate revenue even if every resulting row looked reasonable.

A many-to-one join expects one customer record for each customer ID. `validate='many_to_one'` rejects duplicate keys in the customer table. `indicator=True` identifies orders with no matching customer; a successful join alone does not guarantee complete coverage.

In [ ]:
enriched = clean_orders.merge(customers, on='customer_id', how='left',
                              validate='many_to_one', indicator=True)
assert enriched['_merge'].eq('both').all(), 'Some orders have no customer record'
enriched.drop(columns='_merge')


## 17. Deliver a sales report with reconciled totals

Check that the final report uses the intended date range, grouping and treatment of missing records. Trace one result back to its source rows. The file existing on disk is not evidence that the correct population was exported.

Combine deduplication, type conversion, validated joins, and aggregation. Build one report row per customer tier, check that its totals match the cleaned orders, then export and read it back. As an extension, add an unknown customer ID and inspect the failed coverage check in section 16.

In [ ]:
from pathlib import Path

report = (enriched.groupby('tier', observed=True)
          .agg(order_count=('order_id', 'size'), revenue=('amount', 'sum'))
          .reset_index())
assert report['order_count'].sum() == 4
assert report['revenue'].sum() == clean_orders['amount'].sum() == 400
output_dir = Path('outputs')
output_dir.mkdir(exist_ok=True)
report_path = output_dir / 'pandas_sales_report.csv'
report.to_csv(report_path, index=False)
restored = pd.read_csv(report_path)
assert restored['revenue'].sum() == 400
print(report.to_string(index=False))
print('Saved:', report_path)


## Explain what the evidence supports

Choose one report total and explain its source population, exclusions and reconciliation check. Describe a join or missing-value decision that would change that total and why.

Change one assumption or input at a time. Predict the effect before rerunning, compare it with the actual result, and explain any difference. Keep the original result available so that the comparison is meaningful.

## Second project: reconcile a payment export with a customer snapshot

A payment export contains retries, invalid amounts, malformed dates and unknown customer keys. Refunds are signed negative amounts, so deleting every negative amount would inflate net receipts. Reconcile 6,060 raw rows into 60 retries, 50 rejected events and 5,950 usable events. These are generated operational-style records, not actual customer payments. The generator's clean source is used only as a verification oracle, never to fill invalid source fields. Work within a small laptop-sized budget, then identify which intermediate tables or buffers would grow at scale.

Run these cells in order. This extension uses its own `p2_` variables and writes only its named outputs under `outputs/project2/`. Predict each comparison before running it; the worked discussion after the code explains the result and its limits.

### Work through the duplicate-join trap

Two payment rows worth 100 and 200 cents total 300. If their shared customer appears twice in the dimension table, a join produces four rows and 600 cents. Deduplicating payment rows after this join is not a sound repair for a dimension-key violation. Check the right-hand key and use `validate='many_to_one'` before aggregating.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import time
from IPython.display import display
p2_out=Path('outputs/project2');p2_out.mkdir(parents=True,exist_ok=True)

p2_rng=np.random.default_rng(91)
p2_n=6000
p2_source=pd.DataFrame({'event_id':np.arange(p2_n),'customer_id':p2_rng.integers(1,121,p2_n),
    'day':pd.Timestamp('2025-01-01')+pd.to_timedelta(p2_rng.integers(0,90,p2_n),unit='D'),
    'amount_cents':p2_rng.integers(100,100000,p2_n)})
p2_source.loc[p2_source.event_id%17==0,'amount_cents'] *= -1 # Signed refunds are valid.
p2_raw=p2_source.astype(str)
p2_raw.loc[0:19,'amount_cents']='bad'
p2_raw.loc[20:29,'amount_cents']=''
p2_raw.loc[30:37,'day']='not-a-date'
p2_raw.loc[38:49,'customer_id']='9999'
p2_raw=pd.concat([p2_raw,p2_raw.iloc[-60:]],ignore_index=True)
p2_customers=pd.DataFrame({'customer_id':np.arange(1,121),'region':np.tile(['East','West','North','South'],30)})
p2_raw.to_csv(p2_out/'ledger_raw.csv',index=False)
# Independent generator-based expected population; never repair a bad value from this oracle.
p2_expected=p2_source.iloc[50:]
p2_expected_total=int(p2_expected.amount_cents.sum())
assert len(p2_raw)==6060 and len(p2_expected)==5950

p2_pay=pd.DataFrame({'customer_id':[1,1],'cents':[100,200]})
p2_bad_dim=pd.DataFrame({'customer_id':[1,1],'region':['East','West']})
assert p2_pay.merge(p2_bad_dim).cents.sum()==600
try:p2_pay.merge(p2_bad_dim,validate='many_to_one')
except pd.errors.MergeError:print('Caught conflicting dimension keys before publication.')
else:raise AssertionError('Expected join validation error')
p2_unique=p2_raw.drop_duplicates().copy()
assert p2_unique.event_id.is_unique # Conflicting duplicate IDs would fail here.
p2_unique['cents']=pd.to_numeric(p2_unique.amount_cents,errors='coerce')
p2_unique['parsed_day']=pd.to_datetime(p2_unique.day,errors='coerce')
p2_unique['customer_id']=pd.to_numeric(p2_unique.customer_id,errors='coerce')
p2_good=p2_unique.cents.notna()&p2_unique.parsed_day.notna()&p2_unique.customer_id.isin(p2_customers.customer_id)
p2_clean=p2_unique.loc[p2_good].copy();p2_clean['cents']=p2_clean.cents.astype('int64')
p2_rejected=p2_unique.loc[~p2_good]
p2_enriched=p2_clean.merge(p2_customers,on='customer_id',validate='many_to_one')
assert len(p2_enriched)==5950 and len(p2_rejected)==50
assert p2_enriched.cents.sum()==p2_expected_total

### Work through the computation budget

Compare a row-wise lookup with a vectorized column mapping on the same already-clean data. The goal is identical business meaning before timing. Run three trials and report the median; small timings are machine-specific. The vectorized method avoids a Python callback per row. Both still require the full clean DataFrame in memory.

In [ ]:
p2_region_map=p2_customers.set_index('customer_id').region.to_dict()
p2_times=[]
for p2_method in ['row_apply','column_map']:
    p2_durations=[]
    for _ in range(3):
        p2_t=time.perf_counter()
        p2_regions=(p2_clean.apply(lambda r:p2_region_map[r.customer_id],axis=1)
                    if p2_method=='row_apply' else p2_clean.customer_id.map(p2_region_map))
        p2_totals=p2_clean.assign(region=p2_regions).groupby('region').cents.sum().sort_index()
        p2_durations.append(time.perf_counter()-p2_t)
    p2_times.append({'method':p2_method,'median_seconds':np.median(p2_durations)})
    assert p2_totals.sum()==p2_expected_total
p2_report=p2_enriched.groupby('region').agg(events=('event_id','size'),net_cents=('cents','sum'))
assert p2_report.events.sum()==5950
p2_report.to_csv(p2_out/'pandas_payment_report.csv');p2_rejected.to_csv(p2_out/'pandas_rejected.csv',index=False)
display(p2_report);display(pd.DataFrame(p2_times))

### Explain the result before recommending an optimization

The report includes refunds, excludes unrecoverable fields and reconciles to the known valid source population. Integer cents avoid decimal display being mistaken for exact stored arithmetic. A faster map does not settle how to handle a genuinely conflicting customer snapshot; that needs an effective-date/version policy.

**Try:** add a second row for one event ID with a changed amount. **Worked reasoning:** exact-row deduplication retains both versions, then the uniqueness assertion fails. Do not silently keep the last file row: source row order is not a documented event-version rule. Decide which timestamp/revision is authoritative before extending the pipeline.